In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import plotly.io as pio
import torch
from skimage import transform as sktransform
from torchvision import transforms

from pathlib import Path
project_root = Path(os.path.dirname(os.getcwd()))
sys.path.append(str(project_root.joinpath('src')))

from more_anns.models import AlexNet
from more_anns.losses import get_angle_representation
from more_anns.preprocessing import create_circular_mask

from utils.decoder_functions import decode_all
from utils.equivariance_functions import example_equiv_projs, equivariance_summary_all
from utils.helper import load_var_dim, band_summary_per_stim
from utils.plotting_functions import plot_models_overlay, make_df_extractor, make_row_extractor, \
    plot_rotation_robustness, plot_orientation_robustness
from utils.robustness import load_imagenet_samples, evaluate_rotation_robustness, \
    evaluate_orientation_robustness, circular_error_ordered, DEFAULT_ANGLES

pio.kaleido.scope.mathjax = None

In [ ]:
#the two network families of this notebook (conv1-5 of each network, N = 250 units, d = 20):
#ED Fig 6: AlexNet variants trained on different objectives ('alexnet' = the default classification
#network of the main figures; the trained variants need their checkpoints under
#cache/more_anns/{experiment}/best_model.pth, see docs/ann_training.md)
model_df_anns = pd.read_csv( project_root / 'params' / 'model_params_anns.csv', header=0)
#SI Fig 6: AlexNet-style networks with built-in rotation equivariance (eqv-c1: none, eqv-c15: exact
#in every layer, eqv-c15r: relaxed across layers C15 > C5 > C1); loading them needs the escnn package
model_df_eqv = pd.read_csv( project_root / 'params' / 'model_params_eqv.csv', header=0)
params_df = pd.read_csv( project_root / 'params' / 'fixed_params_anns.csv', header=0, index_col=0)

n_neurons = int(params_df.loc['n_neurons'].value)
d = int(params_df.loc['n_procs'].value)
layers = ['conv1', 'conv2', 'conv3', 'conv4', 'conv5']
results_folder = project_root / 'results_extended_anns'
results_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
#per family: the networks in their left-to-right order within a layer (the overlays dodge the
#networks along x by dx_models, layer spacing = 1), their colours, and the file prefix of the
#family's figures ({prefix}_{quantity}_by_model.pdf in results_extended_anns)
FAMILIES = {
    'more_anns': dict(model_df=model_df_anns,
                      nets=['alexnet', 'alt_alexnet_rot-aug', 'alt_alexnet_multi-task', 'alt_alexnet_ori', 'random_alexnet'],
                      colors={'alexnet':                'rgba(100, 100, 100, 1)',
                              'alt_alexnet_rot-aug':    'rgba(44, 160, 44, 1)',
                              'alt_alexnet_multi-task': 'rgba(214, 39, 40, 1)',
                              'alt_alexnet_ori':        'rgba(227, 119, 194, 1)',
                              'random_alexnet':         'rgba(180, 180, 180, 1)'}),
    'eqv_anns': dict(model_df=model_df_eqv,
                     nets=['eqv-c1', 'eqv-c15r', 'eqv-c15'],
                     colors={'eqv-c1':   'rgba(0, 0, 0, 1)',
                             'eqv-c15r': 'rgba(214, 39, 40, 1)',
                             'eqv-c15':  'rgba(227, 119, 194, 1)'}),
}
dx_models = 0.12

def present_nets(fam):
    return [m for m in fam['nets'] if m in fam['model_df']['network'].values]

### caches: responses, equivariance and decoders of both families
takes a long time to run the first time (every cache is reused afterwards); needs the trained checkpoints under `cache/more_anns/{experiment}/best_model.pth`

In [ ]:
for name, fam in FAMILIES.items():
    #trial-averaged responses and example projections, then the per-subsample invariant / tuned
    #quantification (as plot_figures.ipynb for the mouse areas and the default AlexNet)
    example_equiv_projs(fam['model_df'], params_df, project_root)
    equivariance_summary_all(fam['model_df'], params_df, project_root)
    #decoders in the full space (Fig 1d-e analogues, C = 0.001) and in the estimated invariant /
    #tuned subspaces (Fig 3c analogues, C = 1)
    decode_all(fam['model_df'], params_df, project_root, c_range=[0.001])
    decode_all(fam['model_df'], params_df, project_root, c_range=[1], equivariant=True)

### decoding errors across layers (ED Fig 6f, SI Fig 6d)
the full-space decoders (semi-transparent diamonds) and the same decoders in the estimated invariant (image identity) and tuned (relative orientation) subspaces (opaque circles); mean ± sem across the held-out items

In [ ]:
def collect_decoders(model_df, kind, subspace=False):
    """Per-item decoder results of every (network, layer) of model_df, with 'model' and 'area'
    columns: kind 'class' (image-identity SVM, column 'acc test') or 'ori' (relative orientation,
    column 'res'); subspace=False reads the full-space caches (C = 0.001), subspace=True those of
    the estimated invariant / tuned subspace (C = 1, d dims). Rows without a cache are skipped."""
    sub = f'_{d}dim' if subspace else ''
    frames = []
    for model_name, layer_name in zip(model_df['network'], model_df['layer']):
        if kind == 'class':
            f = project_root / 'cache' / model_name / f'{layer_name}_class2x_decoder_{n_neurons}avg_{1 if subspace else 0.001}C{sub}.csv'
        else:
            f = project_root / 'cache' / model_name / f'{layer_name}_ori2x_decoder_0ref_{n_neurons}avg{sub}.csv'
        if not f.exists():
            print(f'No decoder cache {f.name} for {model_name}, skipping.')
            continue
        tmp_df = pd.read_csv(f)
        tmp_df['model'] = model_name
        tmp_df['area'] = layer_name
        frames.append(tmp_df)
    return pd.concat(frames, axis=0, ignore_index=True) if frames else pd.DataFrame()

In [ ]:
for name, fam in FAMILIES.items():
    nets = present_nets(fam)
    #image identity: ED Fig 6f left / SI Fig 6d left
    # figure stored in project_root / 'results_extended_anns' / f'{name}_decoding_error_by_model.pdf'
    svm_df = collect_decoders(fam['model_df'], 'class')
    svm_sub_df = collect_decoders(fam['model_df'], 'class', subspace=True)
    if not svm_df.empty:
        err = lambda x: 1 - x
        plot_models_overlay(nets, layers, make_df_extractor(svm_df, 'model', 'area', 'acc test', transform=err),
                            ylabel='decoding error (1)', yrange=[0.45, 1.03], y_dp=[1 - 1/91],
                            colors=fam['colors'], dx=dx_models,
                            get_y_err_fn_sub=None if svm_sub_df.empty else make_df_extractor(svm_sub_df, 'model', 'area', 'acc test', transform=err),
                            save_path=results_folder / f'{name}_decoding_error_by_model.pdf')
    #relative orientation: ED Fig 6f right / SI Fig 6d right
    # figure stored in project_root / 'results_extended_anns' / f'{name}_orient_error_by_model.pdf'
    ori_df = collect_decoders(fam['model_df'], 'ori')
    ori_sub_df = collect_decoders(fam['model_df'], 'ori', subspace=True)
    if not ori_df.empty:
        plot_models_overlay(nets, layers, make_df_extractor(ori_df, 'model', 'area', 'res'),
                            ylabel='absolute angle estimation error (deg)', yrange=[0, 100], y_dp=[90],
                            colors=fam['colors'], dx=dx_models,
                            get_y_err_fn_sub=None if ori_sub_df.empty else make_df_extractor(ori_sub_df, 'model', 'area', 'res'),
                            save_path=results_folder / f'{name}_orient_error_by_model.pdf')

### invariant and tuned subspace dimensions, degrees of invariance and tuning (ED Fig 6e, 6g; SI Fig 6c, 6e)
dimension counts and variance-weighted band correlations per fit (neuron subsample × held-out class pair, as Fig 3b and Fig 4b), averaged over subsamples per pair; mean ± sem across the held-out pairs

In [ ]:
def collect_bands(model_df):
    """Per (network, layer) the invariant / tuned dimension counts and the degrees of invariance /
    tuning (variance-weighted band correlations) from the per-subsample var_dim caches
    (helper.band_summary_per_stim). Returns (dim_df, pc_df), one row per (model, area, cat) with
    cat 0 = invariant, 1 = tuned, the value ('dim' / 'pc') and its sem across held-out pairs ('erry')."""
    dim_rows, pc_rows = [], []
    for model_name, layer_name in zip(model_df['network'], model_df['layer']):
        f = project_root / 'cache' / model_name / f'{layer_name}_var_dim_{n_neurons}avg_{d}dim.csv'
        if not f.exists():
            print(f'No quantification cache {f.name} for {model_name}, skipping.')
            continue
        stim_df = band_summary_per_stim(load_var_dim(f), pc_weighted=True)
        n_stim = stim_df['stim'].nunique()
        for cat, band in enumerate(['inv', 'tun']):
            dim_rows.append({'model': model_name, 'area': layer_name, 'cat': cat,
                             'dim': stim_df[f'd_{band}'].mean(), 'erry': stim_df[f'd_{band}'].std() / np.sqrt(n_stim)})
            pc = stim_df[f'pc test {band}'].dropna()
            pc_rows.append({'model': model_name, 'area': layer_name, 'cat': cat,
                            'pc': pc.mean(), 'erry': pc.std() / np.sqrt(n_stim)})
    return pd.DataFrame(dim_rows), pd.DataFrame(pc_rows)

In [ ]:
for name, fam in FAMILIES.items():
    nets = present_nets(fam)
    dim_df, pc_df = collect_bands(fam['model_df'])
    if dim_df.empty:
        continue
    for cat, band in enumerate(['inv', 'tuned']):
        label = 'invariant' if cat == 0 else 'tuned'
        #dimensions: ED Fig 6e / SI Fig 6c (left: invariant, right: tuned)
        # figures stored in project_root / 'results_extended_anns' / f'{name}_{band}_dim_by_model.pdf'
        plot_models_overlay(nets, layers, make_row_extractor(dim_df, 'model', 'cat', cat, 'dim', 'erry', layer_col='area'),
                            ylabel=f'{label} dimension (1)', yrange=[0, 20], colors=fam['colors'], dx=dx_models,
                            save_path=results_folder / f'{name}_{band}_dim_by_model.pdf')
        #degrees of invariance and tuning: ED Fig 6g / SI Fig 6e
        # figures stored in project_root / 'results_extended_anns' / f'{name}_{band}_pcorr_by_model.pdf'
        plot_models_overlay(nets, layers, make_row_extractor(pc_df, 'model', 'cat', cat, 'pc', 'erry', layer_col='area'),
                            ylabel=f'{label} Pearson correlation (1)', yrange=[0, 1], colors=fam['colors'], dx=dx_models,
                            save_path=results_folder / f'{name}_{band}_pcorr_by_model.pdf')

### CC_norm over neuron number and fitted dimensions (SI Fig 6f)
the pooled centered CC_norm of the equivariance model over an (N, d) grid for conv1-5 of the three networks
with built-in equivariance, with the injected noise and (conv1, conv5) noiseless; N up to each layer's
reliably responsive unit count. Needs the single-trial caches of both noise models; the surfaces are long
computations, cached per (N, d) and skipped afterwards

In [ ]:
from utils.equivariance_validation import trial_caches_all
from utils.simulation import ccnorm_surface, reliable_count

params_df_noiseless = pd.read_csv( project_root / 'params' / 'fixed_params_eqv_snr0.csv', header=0, index_col=0)
trial_caches_all(model_df_eqv, params_df, project_root)
trial_caches_all(model_df_eqv, params_df_noiseless, project_root)
for _, r in model_df_eqv.iterrows():
    n_rel = reliable_count(project_root, r['network'], r['layer'])
    n_list = [n for n in (250, 500, 1000, 2000, 4000) if n < n_rel] + [n_rel]
    ccnorm_surface(project_root, r['network'], r['layer'], n_list, n_max=2)
    if r['layer'] in ('conv1', 'conv5'):
        ccnorm_surface(project_root, r['network'], r['layer'], n_list, n_max=2, noise_model='noiseless')

In [ ]:
from utils.plot_simulation import plot_eqv_net_peak_vs_n

#SI Fig 6f: peak over d of CC_norm vs N, conv1 (light) to conv5 (dark) in the network's colour, noisy solid /
#noiseless dashed -> results_extended_anns/ccnorm_peak_vs_N_{eqv-c1,eqv-c15,eqv-c15r}.pdf
for net in FAMILIES['eqv_anns']['nets']:
    plot_eqv_net_peak_vs_n(project_root, net, results_folder=results_folder)

### rotation robustness (ED Fig 6d top, SI Fig 6b)
network output classification error on 100 ImageNet-1k validation images (cached in `cache/imagenet_val_100_seed42.pkl`; the first run downloads them from the gated HuggingFace dataset `ILSVRC/imagenet-1k`) at the 15 stimulus rotations

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ANGLES = DEFAULT_ANGLES
cache_file = project_root / 'cache' / 'imagenet_val_100_seed42.pkl'
samples = load_imagenet_samples(cache_file)

# Pretrained AlexNet (ImageNet)
model_pretrained = AlexNet(task='classification', pretrained_weights='imagenet').to(DEVICE)
model_pretrained.eval()
correct_pretrained = evaluate_rotation_robustness(model_pretrained, samples, ANGLES, DEVICE)

# Random AlexNet
model_random = AlexNet(task='classification', pretrained_weights=None).to(DEVICE)
model_random.eval()
correct_random = evaluate_rotation_robustness(model_random, samples, ANGLES, DEVICE)

# rot-aug: classification-only with rotation augmentation
ckpt_rot_aug = torch.load(project_root / 'cache' / 'more_anns' / 'rot-aug' / 'best_model.pth',
                           map_location='cpu', weights_only=True)
model_rot_aug = AlexNet(task='classification', pretrained_weights=None)
model_rot_aug.load_state_dict(ckpt_rot_aug['model_state_dict'])
model_rot_aug = model_rot_aug.to(DEVICE)
model_rot_aug.eval()
correct_rot_aug = evaluate_rotation_robustness(model_rot_aug, samples, ANGLES, DEVICE)

# multi-task: classification + ordered orientation
ckpt_mt = torch.load(project_root / 'cache' / 'more_anns' / 'multi-task' / 'best_model.pth',
                      map_location='cpu', weights_only=True)
cfg_mt = ckpt_mt['config']
model_mt = AlexNet(
    task='multitask', similarity_layer=cfg_mt['similarity_layer'],
    orientation_fusion=cfg_mt['orientation_fusion'],
    orientation_output_dim=2,
    orientation_activation=cfg_mt.get('orientation_activation', 'linear'),
)
model_mt.load_state_dict(ckpt_mt['model_state_dict'])
model_mt = model_mt.to(DEVICE)
model_mt.eval()
correct_mt = evaluate_rotation_robustness(model_mt, samples, ANGLES, DEVICE, use_backbone=True)

In [ ]:
rotation_results = {
    'random': correct_random,
    'pretrained': correct_pretrained,
    'rot-aug': correct_rot_aug,
    'multi-task': correct_mt,
}
fig = plot_rotation_robustness(rotation_results, ANGLES, project_root, results_folder=results_folder)
fig.show()

In [ ]:
#SI Fig 6b: the three networks with built-in equivariance. Their inputs follow the training pipeline
#of more_anns (Resize 256, CenterCrop 224, rotation with bilinear interpolation and gray fill, circular
#mask, ImageNet normalisation); ED Fig 6d above keeps the evaluation of the AlexNet variants
#(utils.robustness.evaluate_rotation_robustness: rotation with black fill, no mask).
from more_anns.equivariant_models import build_equivariant_alexnet   #needs the escnn package

EQV_NETS = ['eqv-c15', 'eqv-c15r', 'eqv-c1']
_normalise = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
_base = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])
_mask = create_circular_mask(224)[:, :, None]   #(224, 224, 1), 1 inside the disc

def rotate_mask_tensor(img224, angle, use_mask=True):
    """Rotate a 224-px RGB PIL image (bilinear, gray fill), apply the circular mask, normalise."""
    arr = np.asarray(img224, dtype=np.float32)
    if angle != 0:
        arr = sktransform.rotate(arr, angle, mode='constant', cval=128.0, preserve_range=True, order=1)
    if use_mask:
        arr = np.where(_mask == 1, arr, 128.0)
    return _normalise(torch.from_numpy(arr / 255.0).permute(2, 0, 1).float())

def evaluate_eqv_rotation_robustness(network, samples, angles, device):
    """Top-1 correctness of the network for every image x rotation; (n_images, n_angles) bool."""
    ckpt = torch.load(project_root / 'cache' / 'more_anns' / network / 'best_model.pth', map_location='cpu', weights_only=False)
    cfg = ckpt['config']
    model = build_equivariant_alexnet(cfg['backbone'], num_classes=cfg.get('num_classes', 1000))
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device).eval()
    use_mask = cfg.get('use_circular_mask', True)
    correct = np.zeros((len(samples), len(angles)), dtype=bool)
    with torch.no_grad():
        for i, item in enumerate(samples):
            img = item['image']
            if img.mode != 'RGB':
                img = img.convert('RGB')
            img = _base(img)
            batch = torch.stack([rotate_mask_tensor(img, a, use_mask) for a in angles]).to(device)
            (logits,) = model(batch)
            correct[i] = logits.argmax(dim=1).cpu().numpy() == item['label']
    return correct

eqv_rotation_results = {net: evaluate_eqv_rotation_robustness(net, samples, ANGLES, DEVICE) for net in EQV_NETS}
# figure stored in project_root / 'results_extended_anns' / 'eqv_rotation_robustness.pdf'
fig = plot_rotation_robustness(eqv_rotation_results, ANGLES, project_root, results_folder=results_folder,
                               colors=FAMILIES['eqv_anns']['colors'], filename='eqv_rotation_robustness.pdf', showlegend=False)
fig.show()

### orientation robustness (ED Fig 6d bottom)

In [ ]:
rep_ordered = get_angle_representation('sincos_ordered')

# ori: orientation-only (sincos_ordered)
ckpt_ori = torch.load(project_root / 'cache' / 'more_anns' / 'ori' / 'best_model.pth',
                       map_location='cpu', weights_only=True)
cfg_ori = ckpt_ori['config']
model_ori = AlexNet(
    task='orientation', similarity_layer=cfg_ori['similarity_layer'],
    orientation_fusion=cfg_ori['orientation_fusion'],
    orientation_output_dim=2,
    orientation_activation=cfg_ori.get('orientation_activation', 'linear'),
)
model_ori.load_state_dict(ckpt_ori['model_state_dict'])
model_ori = model_ori.to(DEVICE)
model_ori.eval()

# Random baseline: same architecture as ori, random weights
model_orient_random = AlexNet(
    task='orientation', similarity_layer=cfg_ori['similarity_layer'],
    orientation_fusion=cfg_ori['orientation_fusion'],
    orientation_output_dim=2,
    orientation_activation=cfg_ori.get('orientation_activation', 'linear'),
).to(DEVICE)
model_orient_random.eval()

pred_angles_ori = evaluate_orientation_robustness(model_ori, samples, rep_ordered, ANGLES, DEVICE, mode='orientation')
pred_angles_mt = evaluate_orientation_robustness(model_mt, samples, rep_ordered, ANGLES, DEVICE, mode='multitask')
pred_angles_random = evaluate_orientation_robustness(model_orient_random, samples, rep_ordered, ANGLES, DEVICE, mode='orientation')

In [ ]:
true_diff = ANGLES.astype(float)
errors_ori = circular_error_ordered(pred_angles_ori, true_diff[None, :])
errors_mt = circular_error_ordered(pred_angles_mt, true_diff[None, :])
errors_random = circular_error_ordered(pred_angles_random, true_diff[None, :])

orientation_results = {
    'random': errors_random,
    'ori': errors_ori,
    'multi-task': errors_mt,
}
fig = plot_orientation_robustness(orientation_results, ANGLES, project_root, results_folder=results_folder)
fig.show()